# Day 2 — B 담당: Materiality 최종 적용 + 전체 데이터 처리

**좋은 소식 먼저**: PRD 원안엔 '관련성+방향'을 직접 라벨링(150~400건)해서 학습(linear probing)하는 단계가 있었는데, 우리는 이미 학습이 끝난 `KR-FinBert-SC`를 쓰고 있어서 **이 단계 통째로 필요 없다.** Day1에서 감성분류·카테고리분류가 이미 끝났으니, Day2는 그 결과에 **업종별 Materiality(1/0)를 실제로 붙이는 것**이 핵심이다.

**오늘 할 일**
1. A가 만든 종목별 업종(`sector`)을, B가 만든 SASB 11개 업종 Materiality Map에 연결하기 위한 **KRX 업종 → SASB 업종 매핑표** 만들기
2. 매핑표로 뉴스마다 `is_material`(1/0) 최종 계산
3. (시간 남으면) 카테고리 분류 품질 샘플 점검
4. Day1보다 더 많은/전체 뉴스로 파이프라인 재실행
5. 최종 `news_features_day2.csv` 저장 → C에게 전달

In [1]:
!pip install -q transformers torch pandas accelerate

## 0. Day1 산출물 + A의 산출물 불러오기

필요한 파일 2개(없으면 있는 것만이라도):
- Day1에서 만든 `news_features_day1.csv`, `materiality_map_draft.csv`
- A가 만든 `price_features_labeled.csv` (여기 `ticker`, `sector` 컬럼이 있어야 함 — 아직 A가 전달 안 했으면 1~2단계는 건너뛰고 3단계부터 먼저 해도 됨)

In [5]:
from google.colab import files
import pandas as pd

print("news_features_day1.csv 와 materiality_map_draft.csv 를 선택하세요 (여러 개 동시 선택 가능)")
uploaded = files.upload()

news_df = pd.read_csv("news_features_day1 (1).csv")
materiality_map = pd.read_csv("materiality_map_draft.csv")
print(news_df.shape, materiality_map.shape)
news_df.head()

news_features_day1.csv 와 materiality_map_draft.csv 를 선택하세요 (여러 개 동시 선택 가능)


Saving materiality_map_draft.csv to materiality_map_draft.csv
Saving news_features_day1 (1).csv to news_features_day1 (1) (3).csv
(400, 9) (55, 3)


,ticker,date,title,text,description,news_direction,news_severity,news_category,confidence_score
0,LG에너지솔루션,"Mon, 20 Jul 2026 18:16:00 +0900",NaN,LG에너지솔루션은 글로벌배터리연합(GBA)의 배터리 여권 시범사업에 참여해 협력사 ...,NaN,positive,0.972617,ESG,0.569835
1,SK하이닉스,"Mon, 20 Jul 2026 18:38:00 +0900",NaN,행사장에 전시된 히트브랜드 수상 제품 에센코어 클레브 DDR5 6000 CL30 B...,NaN,neutral,0.985198,ESG,0.329917
2,LG에너지솔루션,"Mon, 20 Jul 2026 14:44:00 +0900",NaN,"우리은행, 교보생명, LG전자, LG에너지솔루션 등 관련기업도 참석해 준공을 축하했...",NaN,positive,0.997881,산업·사업동향,0.542388
3,현대차,"Mon, 20 Jul 2026 17:42:00 +0900",NaN,정 회장은 아산테헤네 국왕에게 현대차의 글로벌 자동차 생산 거점과 제조 공정을 담은...,NaN,positive,0.985933,산업·사업동향,0.734117
4,카카오,"Mon, 20 Jul 2026 18:18:00 +0900",NaN,"""AI로 보는 반려동물 속마음""…카카오, '카나나 팻레터 만들기' 오픈",NaN,neutral,0.999881,기타,0.627316


In [13]:
# A의 파일이 준비됐으면 같이 업로드. 없으면 이 셀은 건너뛰고 아래 '수동 매핑' 섹션에서 ticker를 직접 나열해도 된다.
print("price_features_labeled.csv 를 선택하세요 (없으면 취소/스킵)")
uploaded2 = files.upload()
price_df = pd.read_csv(list(uploaded2.keys())[0]) if uploaded2 else None
if price_df is not None:
    print(price_df[["ticker", "sector"]].drop_duplicates())

price_features_labeled.csv 를 선택하세요 (없으면 취소/스킵)


Saving price_features_labeled (5).csv to price_features_labeled (5).csv
      ticker         sector
0        660     Technology
863     5380     Automotive
1726    5490      Materials
2589    5930     Technology
3452   35420  Communication
4315   35720  Communication
5178  373220         Energy


In [7]:
import pandas as pd
from google.colab import files

# 1. 파일 불러오기 (이미 업로드되어 있는 파일들 활용)
price_df = pd.read_csv("price_features_labeled.csv")
materiality_map = pd.read_csv("materiality_map_draft.csv")

print("--- 파일 컬럼 확인 ---")
print("price_df 컬럼:", price_df.columns.tolist())
print("materiality_map 컬럼:", materiality_map.columns.tolist())

# 2. ticker 기준으로 sector 정보 병합하기
if 'ticker' in materiality_map.columns and 'sector' in materiality_map.columns:
    # materiality_map에서 필요한 컬럼만 뽑고 중복 제거
    sector_map = materiality_map[['ticker', 'sector']].drop_duplicates()

    # price_df에 이미 sector가 있다면 지우고 깔끔하게 새로 합침
    if 'sector' in price_df.columns:
        price_df = price_df.drop(columns=['sector'])

    # merge로 묶기
    price_df = pd.merge(price_df, sector_map, on='ticker', how='left')
    print("\n✅ 성공: materiality_map의 sector 정보가 price_features_labeled에 완벽히 합쳐졌습니다!")
else:
    print("\n⚠️ materiality_map_draft.csv 파일 내에 'ticker' 또는 'sector' 컬럼명을 다시 확인해주세요.")

# 3. 합쳐진 파일을 price_features_labeled.csv로 덮어쓰기 저장
output_filename = "price_features_labeled.csv"
price_df.to_csv(output_filename, index=False)
print(f"파일 저장 완료: {output_filename}")

# 4. 컴퓨터로 바로 다운로드 실행
files.download(output_filename)

--- 파일 컬럼 확인 ---
price_df 컬럼: ['ticker', 'date', 'log_return_1d', 'volatility_20d', 'volume_zscore', 'beta_60d', 'label_direction_next_day']
materiality_map 컬럼: ['sasb_sector', 'news_category', 'is_material']

⚠️ materiality_map_draft.csv 파일 내에 'ticker' 또는 'sector' 컬럼명을 다시 확인해주세요.
파일 저장 완료: price_features_labeled.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import pandas as pd
from google.colab import files

# 1. 방금 다운로드받은 price_features_labeled (1).csv 파일 불러오기
# (이름이 길다면 'price_features_labeled.csv'로 바꿔서 실행하셔도 됩니다)
price_df = pd.read_csv("price_features_labeled.csv")

print("1단계 완료: price_df 불러오기 성공 (현재 컬럼:", price_df.columns.tolist(), ")")

# 2. materiality_map_draft.csv 파일 업로드하기
print("\n👉 materiality_map_draft.csv 파일을 선택해주세요:")
uploaded = files.upload()
map_filename = list(uploaded.keys())[0]
materiality_map = pd.read_csv(map_filename)

print("2단계 완료: materiality_map 불러오기 성공 (현재 컬럼:", materiality_map.columns.tolist(), ")")

# 3. ticker 기준으로 sector 정보 합치기 (Merge)
if 'ticker' in materiality_map.columns and 'sector' in materiality_map.columns:
    # 매핑 파일에서 ticker와 sector만 추출 후 중복 제거
    sector_map = materiality_map[['ticker', 'sector']].drop_duplicates()

    # 만약 기존에 sector가 있었다면 제거 후 병합
    if 'sector' in price_df.columns:
        price_df = price_df.drop(columns=['sector'])

    # merge 실행
    price_df = pd.merge(price_df, sector_map, on='ticker', how='left')
    print("\n✅ 성공: materiality_map의 sector 정보가 price_df에 완벽히 합쳐졌습니다!")
else:
    print("\n⚠️ materiality_map 파일에 'ticker' 또는 'sector' 컬럼이 있는지 확인해주세요.")

# 4. 결과 확인 및 저장
print("\n[합쳐진 데이터 상위 5개 미리보기]")
display(price_df.head())

output_file = "price_features_labeled.csv"
price_df.to_csv(output_file, index=False)
print(f"\n파일 저장 완료: {output_file}")

# 5. 컴퓨터로 자동 다운로드
files.download(output_file)

1단계 완료: price_df 불러오기 성공 (현재 컬럼: ['ticker', 'date', 'log_return_1d', 'volatility_20d', 'volume_zscore', 'beta_60d', 'label_direction_next_day'] )

👉 materiality_map_draft.csv 파일을 선택해주세요:


Saving materiality_map_draft.csv to materiality_map_draft (1).csv
2단계 완료: materiality_map 불러오기 성공 (현재 컬럼: ['sasb_sector', 'news_category', 'is_material'] )

⚠️ materiality_map 파일에 'ticker' 또는 'sector' 컬럼이 있는지 확인해주세요.

[합쳐진 데이터 상위 5개 미리보기]


,ticker,date,log_return_1d,volatility_20d,volume_zscore,beta_60d,label_direction_next_day
0,660,2023-01-02,NaN,NaN,NaN,NaN,0
1,660,2023-01-03,-0.001322,NaN,NaN,NaN,1
2,660,2023-01-04,0.068993,NaN,NaN,NaN,1
3,660,2023-01-05,0.004926,NaN,NaN,NaN,1
4,660,2023-01-06,0.020669,NaN,NaN,NaN,1



파일 저장 완료: price_features_labeled.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import pandas as pd
from google.colab import files
import os

# 1. price_features 파일 찾기 또는 업로드
price_files = [f for f in os.listdir('.') if 'price_features_labeled' in f]
if price_files:
    price_filename = price_files[0]
    print(f"기존 가격 파일 발견: {price_filename}")
else:
    print("price_features_labeled.csv 파일을 선택해주세요:")
    uploaded = files.upload()
    price_filename = list(uploaded.keys())[0]

price_df = pd.read_csv(price_filename)

# 2. materiality_map_draft 파일 찾기 또는 업로드
if os.path.exists('materiality_map_draft.csv'):
    map_filename = 'materiality_map_draft.csv'
    print("기존 materiality_map_draft.csv 파일 발견")
else:
    print("materiality_map_draft.csv 파일을 선택해주세요:")
    uploaded_map = files.upload()
    map_filename = list(uploaded_map.keys())[0]

materiality_map = pd.read_csv(map_filename)

# 3. ticker 기준으로 sector 컬럼 병합
if 'ticker' in materiality_map.columns and 'sector' in materiality_map.columns:
    sector_map = materiality_map[['ticker', 'sector']].drop_duplicates()

    if 'sector' in price_df.columns:
        price_df = price_df.drop(columns=['sector'])

    price_df = pd.merge(price_df, sector_map, on='ticker', how='left')
    print("성공: sector 정보가 완벽하게 병합되었습니다.")
else:
    print("오류: materiality_map 파일 내에 'ticker' 또는 'sector' 컬럼이 존재하지 않습니다.")

# 4. 최종 파일 저장 및 자동 다운로드
output_filename = "price_features_labeled.csv"
price_df.to_csv(output_filename, index=False)
print(f"최종 파일 저장 완료: {output_filename}")

files.download(output_filename)

기존 가격 파일 발견: price_features_labeled.csv
기존 materiality_map_draft.csv 파일 발견
오류: materiality_map 파일 내에 'ticker' 또는 'sector' 컬럼이 존재하지 않습니다.
최종 파일 저장 완료: price_features_labeled.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
import pandas as pd
from google.colab import files
import os

# 1. 현재 있는 가격 데이터 파일 불러오기
price_files = [f for f in os.listdir('.') if 'price_features_labeled' in f]
price_filename = price_files[0] if price_files else 'price_features_labeled.csv'
price_df = pd.read_csv(price_filename)

print("불러온 가격 파일:", price_filename)

# 2. 종목코드(ticker)별 업종(sector) 매핑 정보 직접 생성
sector_data = {
    660: 'Technology',      # SK하이닉스 등
    5930: 'Technology',     # 삼성전자 등
    373220: 'Energy',       # LG에너지솔루션
    5380: 'Automotive',     # 현대차 등
    5490: 'Materials',      # POSCO홀딩스 등
    35420: 'Communication', # NAVER
    35720: 'Communication'  # 카카오
}

# ticker 타입을 int로 맞추기
price_df['ticker'] = price_df['ticker'].astype(int)

# 3. sector 컬럼 추가 (매핑)
price_df['sector'] = price_df['ticker'].map(sector_data).fillna('Others')

print("\n✅ 성공: 모든 행에 sector 컬럼이 추가되었습니다!")
print(price_df[['ticker', 'sector']].drop_duplicates())

# 4. 파일 저장 및 자동 다운로드
output_filename = "price_features_labeled.csv"
price_df.to_csv(output_filename, index=False)
print(f"\n파일 저장 완료: {output_filename}")

files.download(output_filename)

불러온 가격 파일: price_features_labeled.csv

✅ 성공: 모든 행에 sector 컬럼이 추가되었습니다!
      ticker         sector
0        660     Technology
863     5380     Automotive
1726    5490      Materials
2589    5930     Technology
3452   35420  Communication
4315   35720  Communication
5178  373220         Energy

파일 저장 완료: price_features_labeled.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 1. KRX 업종 → SASB 업종 매핑표 만들기

**이건 코드가 대신 판단해줄 수 없는 부분이다.** A가 가져온 `sector`(반도체, 화학, 은행 같은 한글 업종명)를, 우리 Materiality Map이 쓰는 SASB 11개 업종 중 어디에 해당하는지 사람이 정해야 한다.

아래는 자주 나오는 업종 기준 **초안 매핑표**다. 위 셀에서 실제 `sector` 고유값을 확인했다면, 거기 없는 업종이 있는지 대조하고 빈칸을 채워라. **애매한 업종은 억지로 넣지 말고, 팀장한테 실제 고유값 리스트를 그대로 전달해서 같이 정하거나 나(Claude)한테 물어봐도 된다.**

In [14]:
# 데이터에 어떤 업종명(sector)들이 들어있는지 출력해보기
print(price_df['sector'].unique())  # 또는 materiality_map['sector'].unique()

['Technology' 'Automotive' 'Materials' 'Communication' 'Energy']


In [15]:
# 왼쪽 = A의 sector 값 (실제 값 확인 후 여기에 추가/수정), 오른쪽 = SASB 11개 업종 중 하나
# 실제 데이터의 5개 업종을 완벽하게 반영한 매핑 딕셔너리
krx_to_sasb = {
    "Technology": "Technology & Communications",
    "Automotive": "Transportation",
    "Materials": "Resource Transformation",
    "Communication": "Technology & Communications",
    "Energy": "Extractives & Minerals Processing"
}


if price_df is not None:
    unmapped = set(price_df["sector"].dropna().unique()) - set(krx_to_sasb.keys())
    if unmapped:
        print("⚠️ 아래 업종은 아직 매핑이 안 됐다. 위 딕셔너리에 추가해라:")
        for s in unmapped:
            print(" -", s)
    else:
        print("모든 업종이 매핑됨")

모든 업종이 매핑됨


## 2. 뉴스마다 최종 `is_material` 계산

순서: 뉴스의 `ticker` → (A의 데이터로) `sector` → (위 매핑표로) `sasb_sector` → (Materiality Map으로) `is_material`

In [20]:
if news_df is not None:
    # 뉴스 데이터의 기업 이름(ticker)을 SASB 업종으로 바로 연결하는 번역기
    company_to_sasb = {
        'LG에너지솔루션': 'Extractives & Minerals Processing',
        'SK하이닉스': 'Technology & Communications',
        '삼성전자': 'Technology & Communications',
        '현대차': 'Transportation',
        '카카오': 'Technology & Communications',
        'NAVER': 'Technology & Communications',
        'POSCO홀딩스': 'Resource Transformation'
    }

    # 1. 뉴스 데이터에 sasb_sector 컬럼 직접 매핑
    news_df['sasb_sector'] = news_df['ticker'].map(company_to_sasb)

    # 2. materiality_map과 병합하여 is_material 계산
    final = news_df.merge(
        materiality_map,
        left_on=["sasb_sector", "news_category"],
        right_on=["sasb_sector", "news_category"],
        how="left",
    )

    # 3. 매핑 안 된 조합은 0(immaterial) 처리
    final["is_material"] = final["is_material"].fillna(0).astype(int)

    missing_sector_count = final["sasb_sector"].isna().sum()
    print(f"업종 매핑 안 된 뉴스 행 수: {missing_sector_count} / {len(final)}")
    display(final.head(10))
else:
    print("news_df가 없습니다.")

업종 매핑 안 된 뉴스 행 수: 0 / 400


,ticker,date,title,text,description,news_direction,news_severity,news_category,confidence_score,sasb_sector,is_material
0,LG에너지솔루션,"Mon, 20 Jul 2026 18:16:00 +0900",NaN,LG에너지솔루션은 글로벌배터리연합(GBA)의 배터리 여권 시범사업에 참여해 협력사 ...,NaN,positive,0.972617,ESG,0.569835,Extractives & Minerals Processing,1
1,SK하이닉스,"Mon, 20 Jul 2026 18:38:00 +0900",NaN,행사장에 전시된 히트브랜드 수상 제품 에센코어 클레브 DDR5 6000 CL30 B...,NaN,neutral,0.985198,ESG,0.329917,Technology & Communications,1
2,LG에너지솔루션,"Mon, 20 Jul 2026 14:44:00 +0900",NaN,"우리은행, 교보생명, LG전자, LG에너지솔루션 등 관련기업도 참석해 준공을 축하했...",NaN,positive,0.997881,산업·사업동향,0.542388,Extractives & Minerals Processing,0
3,현대차,"Mon, 20 Jul 2026 17:42:00 +0900",NaN,정 회장은 아산테헤네 국왕에게 현대차의 글로벌 자동차 생산 거점과 제조 공정을 담은...,NaN,positive,0.985933,산업·사업동향,0.734117,Transportation,0
4,카카오,"Mon, 20 Jul 2026 18:18:00 +0900",NaN,"""AI로 보는 반려동물 속마음""…카카오, '카나나 팻레터 만들기' 오픈",NaN,neutral,0.999881,기타,0.627316,Technology & Communications,0
5,LG에너지솔루션,"Mon, 20 Jul 2026 16:36:00 +0900",NaN,"삼성생명 -8.91%, KB금융 -6.79%, 현대차 -6.12%, LG에너지솔루션...",NaN,negative,0.997611,산업·사업동향,0.749816,Extractives & Minerals Processing,0
6,LG에너지솔루션,"Mon, 20 Jul 2026 15:54:00 +0900",NaN,"삼성전자(-4.31%), SK하이닉스(-4.23%), SK스퀘어(-2.89%), 삼...",NaN,negative,0.999762,산업·사업동향,0.425026,Extractives & Minerals Processing,0
7,삼성전자,"Mon, 20 Jul 2026 19:12:00 +0900",NaN,기술통 이청 대표를 중심으로 삼성전자를 비롯한 주요 고객사에 없어서는 안 될 존재가...,NaN,negative,0.733185,산업·사업동향,0.816416,Technology & Communications,0
8,삼성전자,"Mon, 20 Jul 2026 18:24:00 +0900",NaN,삼성전자와 SK하이닉스가 코스피 시가총액의 절반 이상을 차지하는 구조 자체가 변동성...,NaN,neutral,0.963680,기타,0.481938,Technology & Communications,0
9,LG에너지솔루션,"Mon, 20 Jul 2026 17:32:00 +0900",NaN,"현대차(-6.12%), LG에너지솔루션(-4.94%), 삼성바이오로직스(-3.65%...",NaN,negative,0.999466,산업·사업동향,0.390997,Extractives & Minerals Processing,0


## 3. (선택, 시간 남으면) 카테고리 분류 품질 샘플 점검

20개 정도 무작위로 뽑아서 `news_category`가 상식적으로 맞는지 눈으로 확인한다. 이상한 게 많으면 Day1 노트북의 `candidate_labels` 문구를 바꿔서(예: "ESG 관련" → "환경·사회·지배구조(ESG) 관련") 다시 돌려본다.

In [22]:
# 현재 데이터프레임의 모든 컬럼 이름 확인하기
print("전체 컬럼 목록:", final.columns.tolist())

# title 대신 text나 description 등 내용이 들어있는 컬럼을 함께 출력
# (만약 텍스트 컬럼명이 다르면 아래 코드가 알아서 본문 일부를 보여줍니다)
text_col = 'text' if 'text' in final.columns else ('description' if 'description' in final.columns else 'ticker')

sample = final.sample(min(10, len(final)), random_state=1)
display(sample[[text_col, 'news_category', 'news_direction', 'confidence_score']])

전체 컬럼 목록: ['ticker', 'date', 'title', 'text', 'description', 'news_direction', 'news_severity', 'news_category', 'confidence_score', 'sasb_sector', 'is_material']


,text,news_category,news_direction,confidence_score
398,배 대표가 함께 공개한 분석 자료에 따르면 올 5월 27일부터 이달 16일까지 SK...,실적·재무,negative,0.851620
125,삼성전자와 SK하이닉스 등 다른 글로벌 반도체 기업도 수백조원 단위 설비 투자에 뛰...,산업·사업동향,positive,0.701158
328,해당 의원은 네이버 1784 사옥에서 임직원의 건강관리를 제공하는 동시에 네이버웍스...,기타,neutral,0.608515
339,퍼퓨머 에이치가 카카오 선물하기에 입점했다,ESG,neutral,0.506929
172,"POSCO홀딩스(81.3%), 삼성SDI(76.0%), 포스코퓨처엠(69.1%), ...",산업·사업동향,negative,0.392283
342,△삼성생명(-8.91%) △KB금융(-6.79%) △현대차(-6.12%) △LG에너...,산업·사업동향,negative,0.644655
197,"‘역사적 저점’ 카카오, 반등 올까 카카오는 본업에서의안정적 실적에도 불구하고 인공...",산업·사업동향,positive,0.711386
291,삼성전자와 SK하이닉스는 장 초반 저가 매수세에 힘입어 잠시 상승 전환하기도 했지만...,기타,negative,0.373805
29,1% 손실을 냈다. 이론상 수익률과 실제 성과 간 괴리는 66.9%포인트를 나타냈다...,실적·재무,negative,0.859520
284,"수익률 상위 1% 투자자의 순매도 1위는 SK하이닉스였고, LG에너지솔루션과 대덕전...",실적·재무,negative,0.448505


직접 눈으로 보고 이상한 행이 있으면 아래에 메모:
- (예시) "OO기업 신제품 출시" → `기타`로 분류됐는데 `산업·사업동향`이 맞아 보임 → candidate_labels 문구 조정 필요
- "퍼퓸 에이치가 카카오 선물하기에 입점했다" → ESG 로 분류됐는데 산업·사업동향 이 맞아 보임 → candidate_labels 문구 조정 필요
- "삼성전자와 SK하이닉스는 장 초반 저가 매수에 힘입어..." (주가 및 시황 관련) → 기타로 분류됐는데 산업·사업동향이 맞아 보임 → candidate_labels 문구 조정 필요

## 4. 전체 뉴스로 재실행 (Day1이 일부 샘플이었다면)

Day1 노트북에서 만든 `sentiment_pipe`, `category_pipe`를 그대로 재사용해서, 아직 처리 안 한 나머지 뉴스에 대해 3~4번(Day1 노트북) 셀을 다시 돌리면 된다. 이미 전체 뉴스를 다 처리했다면 이 단계는 건너뛰어도 됨.

## 5. 최종 저장 → C에게 전달

In [23]:
output_cols = ["ticker", "date", "title", "news_direction", "news_severity", "news_category", "is_material", "confidence_score"]
output_cols = [c for c in output_cols if c in final.columns]

final[output_cols].to_csv("news_features_day2.csv", index=False, encoding="utf-8-sig")

from google.colab import files as gfiles
gfiles.download("news_features_day2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
# 만약 text 컬럼에 내용이 있다면 빈 title 컬럼을 채우기
if 'text' in final.columns:
    final['title'] = final['title'].fillna(final['text'])

# 다시 올바르게 저장하기
output_cols = ["ticker", "date", "title", "news_direction", "news_severity", "news_category", "is_material", "confidence_score"]
output_cols = [c for c in output_cols if c in final.columns]
final[output_cols].to_csv("news_features_day2.csv", index=False, encoding="utf-8-sig")

print("title 채우기 및 파일 재저장 완료!")

title 채우기 및 파일 재저장 완료!


In [25]:
from google.colab import files
files.download("news_features_day2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 오늘 끝나면 체크할 것
- [ ] `krx_to_sasb` 매핑표에서 '⚠️ 매핑 안 됨' 경고가 안 뜨는지 확인 (안 뜨면 전부 매핑된 것)
- [ ] `is_material` 컬럼이 전부 0 또는 1로 채워졌는지 (NaN 없는지) 확인
- [ ] 샘플 20개 눈으로 봤을 때 카테고리 분류가 크게 이상하지 않은지
- [ ] `news_features_day2.csv`에 `is_material` 컬럼까지 포함됐는지 확인 (Day1 파일엔 없었음)
- [ ] C에게 `news_features_day2.csv` 전달 — 이제 C의 `join_features.py`가 더미 대신 이 파일을 쓸 수 있음